# EIBP Slice Builder: GraphML Parsing
# PTP Testing 

This works in tandem with the Graph Analyzer code previously written, if desired (not required). That code can be found here: https://github.com/pjw7904/Graph-Analyzer/tree/develop

Any graphml file comprised of basic node and edge tags will be able to work with this parser. Each node will be set up as a FABRIC node and any edge will be set up as a l2network between the specified nodes. Currently, this code does not consider extended LANs with more than two nodes.

This was written to take advantage of existing graphml files, as opposed to the FABRIC-enhanced graphml RSPEC files that contain hardware properties that go beyond the basic topological information.

In [9]:
from ipaddress import ip_address, IPv4Address, IPv4Network
import ipaddress
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
try:
    fablib = fablib_manager()
                     
    fablib.show_config()
except Exception as e:
    print(f"Exception: {e}")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,787adfc9-d37e-42f2-8efe-8e32793e0bb8
Bastion Host,bastion.fabric-testbed.net
Bastion Username,tm3886_0000190412
Bastion Private Key File,/home/fabric/work/fabric_config/Fabric_Bastion_Key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key
Sites to avoid,


In [10]:
slice_name=f"Slice for KNIT7 Precision Timing Tutorial"
sites = []
avoid_sites = []
sites = fablib.get_random_sites(count=4,filter_function=lambda x:x['ptp_capable'] is True, avoid=(avoid_sites))

print (f"PTP Capable sites selected are {sites}")

PTP Capable sites selected are ['SRI', 'HAWI', 'EDC', 'MASS']


## Input Required Information

| Variable | Use |
| --- | --- |
| SLICE_NAME    | Name of slice you want to create. Please make sure a slice with that name does not already exist. |
| SITE_NAME     | Name of the FABRIC site you want the nodes to be reserved at. This code does not consider inter-site situations, the entire topology is reserved on a single slice. |
| GRAPH_PATH    | Path to the graphml file you want to use to create a topology. |
| HAS_CLIENTS   | Enter True if clients are present in topology, if not, False. These nodes and the networks connecting them utilizes alternative naming and addressing structures. |
| CLIENT_PREFIX | The naming prefix given to each node (currently, this is required if the topology does have clients) |
| MEAS_ADD      | Enter True if measurements are to be taken on the slice. This requires the inclusion of a separate measurement node |

In [11]:
SLICE_NAME = "Eibp_PTPLARGE13"
SITE_NAME = "AMST"
GRAPH_PATH = "/home/fabric/work/EIBP/Scripts/graphs/eibp_large.graphml"
HAS_CLIENTS = True
CLIENT_PREFIX = "ipnode"
MEAS_ADD = False

## Import the FABlib Library and Confirm the Configuration is Correct

In [12]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

try:
    fablib = fablib_manager()

    fablib.show_config()
except Exception as e:
    print(f"Exception: {e}")

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,787adfc9-d37e-42f2-8efe-8e32793e0bb8
Bastion Host,bastion.fabric-testbed.net
Bastion Username,tm3886_0000190412
Bastion Private Key File,/home/fabric/work/fabric_config/Fabric_Bastion_Key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key
Sites to avoid,


## Parse the GraphML for the Topology and Create the Slice

The minidom library is used to parse the graphml. It assumes proper use of the node and edge tags. Examples of valid tags can be seen below. The name of the node is the node's id. Furthermore, it does not matter which node is the source and destination, it is parsed as an undirected graph and there will be issues if you try and create another network between the same two nodes.

    <node id="L1" />
    <node id="S1" />
    <edge source="L1" target="S1" />


In [13]:
import xml.dom.minidom
from collections import Counter
try:
    #Create the slice
    slice = fablib.new_slice(name=SLICE_NAME)

    # Create dictionary to store nodes
    nodeDict = {}

    # Create dictionary to store information to dump into a file
    logFile = {"name": SLICE_NAME, "site": SITE_NAME, "hasClients": HAS_CLIENTS, "meas": MEAS_ADD}

    # Use XML parser to parse the GraphML file
    docs = xml.dom.minidom.parse(GRAPH_PATH)

    # Find all nodes via the node tag, add each to the slice with Rocky Linux as its base
    nodes = docs.getElementsByTagName("node")
    for node in nodes:
        # Grab the node name and determine if it is a client node based on its name prefix
        nodeName = node.getAttribute("id")
        isClient = True if HAS_CLIENTS and nodeName.startswith(CLIENT_PREFIX) else False # Check for compute/client nodes

        ### LOG FILE INFO
        logFile[nodeName] = {"isClient": isClient, "ssh": None, "networks": {}}

        # Add node to the slice
        nodeInfo = slice.add_node(name=nodeName, cores=4, ram=8, image='default_rocky_8', site=SITE_NAME)

        # nodeDict = 0 -> FABRIC node object, 1 -> is a client/server (True) or not (False)
        nodeDict[nodeName] = {"nodeInfo": nodeInfo, "isClient": isClient}

        print(f'Added node {nodeName}')

    # Find all edges via the edge tag, add each to the slice via an L2Bridge connecting the node interfaces
    edges = docs.getElementsByTagName("edge")
    for edge in edges:
        # grab nodes x and y in edge (x,y)
        source = edge.getAttribute("source")
        target = edge.getAttribute("target")

        # Create an interface name for each interface in the network
        sourceIntfName = f"intf-{target}"
        targetIntfName = f"intf-{source}"

        # Name a network based on if it is a user-facing LAN (edge) or P2P links in the core of the network (core)
        networkPrefix = "edge" if nodeDict[source]["isClient"]  or nodeDict[target]["isClient"] else "core"
        networkName = f'{networkPrefix}-{source}-{target}'

        # Add a NIC for each node that is a part of the edge
        sourceIntf = nodeDict[source]["nodeInfo"].add_component(model='NIC_Basic', name=sourceIntfName).get_interfaces()[0]
        targetIntf = nodeDict[target]["nodeInfo"].add_component(model='NIC_Basic', name=targetIntfName).get_interfaces()[0]

        # Add a L2 network between the interfaces
        slice.add_l2network(name=networkName, interfaces=[sourceIntf, targetIntf], type="L2Bridge")

        ### LOG FILE INFO
        logFile[source]["networks"][networkName] = {"neighbor": target}
        logFile[target]["networks"][networkName] = {"neighbor": source}

        print(f'Added edge {source}-{target}')

except Exception as e:
    print(f"Exception: {e}")

Added node C1
Added node C2
Added node C3
Added node D1
Added node D2
Added node D3
Added node D4
Added node D5
Added node A1
Added node A2
Added node A3
Added node A4
Added node A5
Added node ipnode-1
Added node ipnode-2
Added node ipnode-3
Added node ipnode-4
Added node ipnode-5
Added edge C1-C2
Added edge C1-C3
Added edge C1-D1
Added edge C1-D2
Added edge C1-D3
Added edge C2-C3
Added edge C2-D3
Added edge C2-D4
Added edge C2-D5
Added edge C3-D1
Added edge C3-D2
Added edge C3-D4
Added edge C3-D5
Added edge D1-A1
Added edge D1-A2
Added edge D2-A1
Added edge D2-A3
Added edge D3-A2
Added edge D3-A4
Added edge D4-A3
Added edge D4-A5
Added edge D5-A4
Added edge D5-A5
Added edge A1-ipnode-1
Added edge A2-ipnode-2
Added edge A3-ipnode-3
Added edge A4-ipnode-4
Added edge A5-ipnode-5


## Submit the Slice

In [14]:
%%time
import json

try:
    # Submit Slice Request
    print(f'Submitting the new slice, "{SLICE_NAME}"...')
    slice.submit()
    print(f'{SLICE_NAME} creation done.')

except Exception as e:
    print(f"Slice Fail: {e}")
    traceback.print_exc()


Retry: 18, Time: 2683 sec


ID,b404fe14-e277-43d8-83d9-a112cf1f26a8
Name,Eibp_PTPLARGE13
Lease Expiration (UTC),2025-03-01 17:26:17 +0000
Lease Start (UTC),2025-02-28 17:26:17 +0000
Project ID,787adfc9-d37e-42f2-8efe-8e32793e0bb8
State,StableOK


ID,Name,Cores,RAM,Disk,Image,Image Type,Host,Site,Username,Management IP,State,Error,SSH Command,Public SSH Key File,Private SSH Key File
3008b2b6-760c-4922-bcad-2f4c67621d8b,A1,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fe0f:7ed,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fe0f:7ed,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
f9e088ff-ea65-4e53-9c24-efacf8b7e773,A2,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fe39:4e1a,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fe39:4e1a,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
19702799-c506-4cbf-986a-0f27544fdb6e,A3,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fe2b:e2ae,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fe2b:e2ae,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
0a311c41-52e3-49e4-a7d2-ecb34be58955,A4,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fee9:78b3,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fee9:78b3,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
0bd0246c-ee26-4cb1-83ec-16db8f6ca4e9,A5,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fe8b:b61d,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fe8b:b61d,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
ca78aac8-cde6-4a07-88e1-b2e79b0d0871,C1,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fe2c:43bb,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fe2c:43bb,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
f14573f6-a29f-4ccc-97c5-e090c9c637b0,C2,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fe4c:a012,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fe4c:a012,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
9f739429-ff86-40e1-ab4c-d358987ec469,C3,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fe11:a515,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fe11:a515,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
4e436de8-23e8-40dc-9ac7-4b75c0fdb6c6,D1,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:fe1f:6285,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:fe1f:6285,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key
8b372f87-adc5-46e8-a51a-26b2eff68f65,D2,4,8,10,default_rocky_8,qcow2,amst-w2.fabric-testbed.net,AMST,rocky,2001:610:2d0:fabc:f816:3eff:feaf:9863,Active,,ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config rocky@2001:610:2d0:fabc:f816:3eff:feaf:9863,/home/fabric/work/fabric_config/slice_key.pub,/home/fabric/work/fabric_config/slice_key


ID,Name,Layer,Type,Site,Subnet,Gateway,State,Error
476c4b94-8e3d-426a-87af-cf1247d1c0ba,core-C1-C2,L2,L2Bridge,AMST,None,None,Active,
c2168fc4-655e-420b-8c84-c950dd8dc25d,core-C1-C3,L2,L2Bridge,AMST,None,None,Active,
e5fa9170-b826-4893-a53b-91096da031ac,core-C1-D1,L2,L2Bridge,AMST,None,None,Active,
d784c5e1-c7a9-48eb-9a1b-85e57da63670,core-C1-D2,L2,L2Bridge,AMST,None,None,Active,
c05a630a-d241-462e-98b3-49fa68a6fea0,core-C1-D3,L2,L2Bridge,AMST,None,None,Active,
3d76e054-3ed8-4e9e-afd9-2c47f636bced,core-C2-C3,L2,L2Bridge,AMST,None,None,Active,
e388fc32-9763-48f5-82c0-ba28e394d1bf,core-C2-D3,L2,L2Bridge,AMST,None,None,Active,
1542c860-7255-4abb-89c6-74387c74d905,core-C2-D4,L2,L2Bridge,AMST,None,None,Active,
cdb8c69f-5aa0-4e33-bd9b-d41a7ca3c943,core-C2-D5,L2,L2Bridge,AMST,None,None,Active,
6487991a-b3da-45b5-8cae-e2668e77f7d5,core-C3-D1,L2,L2Bridge,AMST,None,None,Active,


Name,Short Name,Node,Network,Bandwidth,Mode,VLAN,MAC,Physical Device,Device,IP Address,Numa Node,Switch Port
C1-intf-C2-p1,p1,C1,core-C1-C2,100,config,,06:94:D5:68:43:2B,eth2,eth2,fe80::cab4:edc6:f621:5d8,4,HundredGigE0/0/0/7
C1-intf-D2-p1,p1,C1,core-C1-D2,100,config,,16:09:E0:EC:F7:F2,eth5,eth5,fe80::be3f:113:dfce:938,4,HundredGigE0/0/0/7
C1-intf-D1-p1,p1,C1,core-C1-D1,100,config,,0E:EA:F8:31:B9:58,eth3,eth3,fe80::2f3a:f69:2580:6817,4,HundredGigE0/0/0/7
C1-intf-C3-p1,p1,C1,core-C1-C3,100,config,,02:A4:6F:65:1B:1F,eth1,eth1,fe80::823c:e4d8:fbab:8df5,4,HundredGigE0/0/0/7
C1-intf-D3-p1,p1,C1,core-C1-D3,100,config,,12:6D:67:78:0D:28,eth4,eth4,fe80::d505:5f6c:4297:4267,4,HundredGigE0/0/0/7
C2-intf-D4-p1,p1,C2,core-C2-D4,100,config,,1A:1F:7F:E1:3B:30,eth2,eth2,fe80::2ba4:b0be:1bcf:7db6,4,HundredGigE0/0/0/7
C2-intf-D3-p1,p1,C2,core-C2-D3,100,config,,1E:7F:6D:59:7D:8A,eth4,eth4,fe80::c220:2dd8:48a8:f1f6,4,HundredGigE0/0/0/7
C2-intf-C1-p1,p1,C2,core-C1-C2,100,config,,1A:1A:07:E3:A2:56,eth1,eth1,fe80::5b3e:8c26:e026:f6e6,4,HundredGigE0/0/0/7
C2-intf-C3-p1,p1,C2,core-C2-C3,100,config,,1E:17:F8:12:AD:0A,eth3,eth3,fe80::c57b:fe25:6b45:13cc,4,HundredGigE0/0/0/7
C2-intf-D5-p1,p1,C2,core-C2-D5,100,config,,22:87:25:1A:1C:19,eth5,eth5,fe80::1d7a:30cf:6ce5:953c,4,HundredGigE0/0/0/7



Time to print interfaces 3304 seconds
Eibp_PTPLARGE13 creation done.
CPU times: user 40min 21s, sys: 11.7 s, total: 40min 33s
Wall time: 55min 11s


In [15]:
nodes = slice.get_nodes()
for node in nodes:
    print (f"{node.get_name()} is hosted on {node.get_host()}")
    ad = fablib.get_site_advertisement(node.get_site())
    print (f"PTP Capable: { ad.flags.ptp}\n")

C1 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

C2 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

C3 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

D1 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

D2 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

D3 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

D4 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

D5 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

A1 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

A2 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

A3 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

A4 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

A5 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

ipnode-1 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

ipnode-2 is hosted on amst-w2.fabric-testbed.net
PTP Capable: True

ipnode-3 is hosted on amst-w2.fabric-testbed.net
PTP Capab

## Add Basic IPv4 Addressing (optional)

We run this for testing purpose.

In this system, 192.168.0.0/16 is the address space for all interfaces on the FABRIC slice.

Each network is a /24 subnet of this network. Edge networks have client/compute devices with lower address (ex: .1) and networking nodes with higher addresses.

In [16]:
from ipaddress import IPv4Network

def updateMeasNetworkName(node, nodeName, intfName):
    if "meas" not in nodeName:
        node.execute(command=f"sudo ip link set dev {intfName} down")
        node.execute(command=f"sudo ip link set dev {intfName} name meas")
        node.execute(command=f"sudo ip link set dev meas up")

        print(f"\t{nodeName} {intfName} renamed meas")
    else:
        print(f"\tMeasurement node not modified")

    return

# Start with a 1 in the third octet
third_octet = 1

# For each network in the slice
for network in slice.get_networks():
    # Determine if the current network is an edge network
    network_name = network.get_name()
    is_edge_network = network_name.startswith("edge")

    # Print configuration information for edge networks
    if is_edge_network:
        network_address = f'192.168.{third_octet}.0/24'
        current_ip_network = IPv4Network(network_address)
        host_ip_list = list(current_ip_network.hosts())  # Exclude network and broadcast addresses

        print(f"Configuring network {network_name} with IPv4 Network {network_address}")
        third_octet += 1
    # For measurement networks, print configuration without network address
    elif "meas" in network_name:
        print(f"Configuring network {network_name}")

    # For each interface in the network
    for intf in network.get_interfaces():
        intf_name = intf.get_physical_os_interface_name()
        node = intf.get_node()
        node_name = node.get_name()

        # Check if the node is a client node and not a measurement node
        if is_edge_network and "meas" not in network_name:
            # If this is an edge network, assign IP address only to client nodes
            if nodeDict[node_name]["isClient"]:
                current_ipv4_address = host_ip_list.pop(0)
            else:
                current_ipv4_address = host_ip_list.pop()

            # Add the address to the node
            intf.ip_addr_add(addr=current_ipv4_address, subnet=current_ip_network)
            print(f"\t{node_name} {intf.get_device_name()} = {current_ipv4_address}")

            # Update log file information
            logFile[node_name]["networks"][network_name]["subnet"] = str(current_ip_network)
            logFile[node_name]["networks"][network_name]["ipv4"] = str(current_ipv4_address)

        # If it's a measurement network, update interface names
        elif "meas" in network_name:
            updateMeasNetworkName(node, node_name, intf_name)

Configuring network edge-A1-ipnode-1 with IPv4 Network 192.168.1.0/24
	ipnode-1 eth1 = 192.168.1.1
	A1 eth2 = 192.168.1.254
Configuring network edge-A2-ipnode-2 with IPv4 Network 192.168.2.0/24
	A2 eth2 = 192.168.2.254
	ipnode-2 eth1 = 192.168.2.1
Configuring network edge-A3-ipnode-3 with IPv4 Network 192.168.3.0/24
	A3 eth2 = 192.168.3.254
	ipnode-3 eth1 = 192.168.3.1
Configuring network edge-A4-ipnode-4 with IPv4 Network 192.168.4.0/24
	A4 eth2 = 192.168.4.254
	ipnode-4 eth1 = 192.168.4.1
Configuring network edge-A5-ipnode-5 with IPv4 Network 192.168.5.0/24
	A5 eth1 = 192.168.5.254
	ipnode-5 eth1 = 192.168.5.1


## Log Topology Information

In [17]:
for node in slice.get_nodes():
    nodeName = node.get_name()

    if("meas" not in nodeName):
        logFile[nodeName]["ssh"] = node.get_ssh_command()

## LOG FILE INFO
with open(f"{SLICE_NAME}_slice_log.json", "w") as outfile:
    json.dump(logFile, outfile)

## Install and setup linuxptp package on nodes
Download the Ansible role to configure and install the LinuxPTP software. For more details regarding the steps performed in the playbook, please refer to the repo at https://github.com/fabric-testbed/ptp

In [18]:
pre_requisites = None

# Set Deployment tool repository details
repo_branch = 'main'
repo_name = 'ptp'
destination_folder = f"""/tmp/{repo_name}-{repo_branch}"""
clone_instructions = f"""
cd /tmp/;rm -rf /tmp/{repo_name}-{repo_branch};git clone --branch {repo_branch} https://github.com/fabric-testbed/{repo_name}.git {destination_folder};
"""

### Setting PTP Install Restrictions

* If you do not want all interfaces synchronized to PTP, add the name of interfaces to avoid as shown
* Management interfaces are not considered and are avoided by default
* If you do not want the system clock synchronized to PTP set the 'SYNC_SYSTEM_CLOCK' to False
* If you do not have any restrictions for a node, you can omit that node from the list

Example:
```
NODE_RESTRICTIONS = { 
   'node1' : { 'AVOID_IFACES': ['enp6s0'],'SYNC_SYSTEM_CLOCK': False},
   'node2' : { 'AVOID_IFACES': ['enp6s0','enp7s0']},
}
```

In [19]:
NODE_RESTRICTIONS = {}

### Restrict Ansible operation based on tags

* Possible values are ptp_stop,ptp_start,ptp_install 
* Only one tag is allowed
* If empty then all three are performed in the right sequence
* If NODE_RESTRICTIONS are applied along with the tags, the operations will not be performed on the AVOIDED INTERFACES

Example
```
ansible_tags = 'ptp_stop'
```

In [20]:
ansible_tags = ''

### Run Ansible playbook on each node

In [21]:
# Instruction to run ansible command from the node
ansible_instructions = f"""
cd {destination_folder}/ansible;ansible-playbook --connection=local --inventory 127.0.0.1, --limit 127.0.0.1 playbook_fabric_experiment_ptp.yml"""

#Create execute threads
execute_threads = {}

for node in nodes:
    if [ele for ele in ["rocky", "centos"] if (ele in node.get_image())]:
        pre_requisites = f"""
        sudo dnf -y install epel-release ; sudo dnf -y install ansible git;
        """
    elif [ele for ele in ["ubuntu", "debian"] if (ele in node.get_image())]:
        pre_requisites = f"""sudo apt-get update;sudo apt-get -y install ansible git;"""
    else:
        pre_requisites = None
    node_name = node.get_name()
    
    # Create JSON files for extra params that will be provided to ansible
    if node_name in NODE_RESTRICTIONS.keys():    
        extra_ansible_params = f""" --extra-vars @parameters.json""";
        with open('/tmp/'+node_name+'-parameters.json', 'w') as f:
            json.dump(NODE_RESTRICTIONS[node_name], f)
        print (f"Uploading install restrictions for {node_name}")    
        node.upload_file('/tmp/'+node_name+'-parameters.json',destination_folder+'/ansible/parameters.json')
    else:
        extra_ansible_params = ''
    if ansible_tags != '':
        extra_ansible_params = extra_ansible_params + ' --tags '+ansible_tags
        
    print (f"Running the PTP Deployment Ansible Playbook on {node.get_name()}")
    execute_threads[node] = node.execute_thread(\
                f"{pre_requisites}"\
                f"{clone_instructions}"\
                f"{ansible_instructions}"\
                f"{extra_ansible_params}",\
                output_file=f"/tmp/{node.get_name()}_ptpinstall.log"\
                )

    #Wait for results from threads
for node,thread in execute_threads.items():
    print(f"Waiting for result from node {node.get_name()}")
    stdout,stderr = thread.result()

print (f"Ansible Playbook run on all nodes completed\n")

Running the PTP Deployment Ansible Playbook on C1
Running the PTP Deployment Ansible Playbook on C2
Running the PTP Deployment Ansible Playbook on C3
Running the PTP Deployment Ansible Playbook on D1
Running the PTP Deployment Ansible Playbook on D2
Running the PTP Deployment Ansible Playbook on D3
Running the PTP Deployment Ansible Playbook on D4
Running the PTP Deployment Ansible Playbook on D5
Running the PTP Deployment Ansible Playbook on A1
Running the PTP Deployment Ansible Playbook on A2
Running the PTP Deployment Ansible Playbook on A3
Running the PTP Deployment Ansible Playbook on A4
Running the PTP Deployment Ansible Playbook on A5
Running the PTP Deployment Ansible Playbook on ipnode-1
Running the PTP Deployment Ansible Playbook on ipnode-2
Running the PTP Deployment Ansible Playbook on ipnode-3
Running the PTP Deployment Ansible Playbook on ipnode-4
Running the PTP Deployment Ansible Playbook on ipnode-5
Waiting for result from node C1
Waiting for result from node C2
Waitin

In [22]:
slice = fablib.get_slice(name = SLICE_NAME)
nodes = slice.get_nodes()
for node in nodes:
    nodeName = node.get_name()
    node_interfaces = node.get_interfaces()
    for interface_obj in node_interfaces:
        interface = interface_obj.get_device_name()
        print (f"Working on {nodeName}->{interface}")
        print (f" Starting PTP Synchronization on node->{interface}") 
        stdout,stderr = node.execute(f'sudo systemctl start phc2sys@{interface}.service;sleep 5')
        print (f" Get Time from node->{interface} CLOCK/PHC")
        stdout,stderr = node.execute("sudo ethtool -T "+interface+"|grep 'PTP Hardware Clock:'|awk '{print $4}'",quiet=True)
        ptp_index = stdout.strip()
        stdout,stderr = node.execute(f"sudo phc_ctl /dev/ptp{ptp_index} get;sudo phc_ctl /dev/ptp{ptp_index} cmp")
    print (f"Time Sync Operation Completed\n\n")

Working on C1->eth1
 Starting PTP Synchronization on node->eth1
 Get Time from node->eth1 CLOCK/PHC
phc_ctl[3843.536]: clock time is 1740767659.148013762 or Fri Feb 28 18:34:19 2025

phc_ctl[3843.545]: offset from CLOCK_REALTIME is 391ns

Working on C1->eth4
 Starting PTP Synchronization on node->eth4
 Get Time from node->eth4 CLOCK/PHC
phc_ctl[3853.979]: clock time is 1740767669.591499353 or Fri Feb 28 18:34:29 2025

phc_ctl[3853.989]: offset from CLOCK_REALTIME is 4745ns

Working on C1->eth2
 Starting PTP Synchronization on node->eth2
 Get Time from node->eth2 CLOCK/PHC
phc_ctl[3864.572]: clock time is 1740767680.184328652 or Fri Feb 28 18:34:40 2025

phc_ctl[3864.583]: offset from CLOCK_REALTIME is 395ns

Working on C1->eth5
 Starting PTP Synchronization on node->eth5
 Get Time from node->eth5 CLOCK/PHC
phc_ctl[3875.238]: clock time is 1740767690.850342741 or Fri Feb 28 18:34:50 2025

phc_ctl[3875.249]: offset from CLOCK_REALTIME is 207ns

Working on C1->eth3
 Starting PTP Synchroni